# INVESTIGACION - RETO_1 🥼💪

• Investigación : Crear una IA reconocimiento de voz

• El alumno se grabará un video de 5 minutos con su rostro cámara encendida y
explicación del código a detalle y como funciona

• Además, creara un documento en Word : caratura y código fuente.

• Deberá de reconocer su voz en los siguientes idiomas : Ingles y Español

• Una vez que reconozca su voz deberá de buscar una imagen en internet o un articulo
especifico.

In [ ]:
!pip install SpeechRecognition -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 48.0 MB/s eta 0:00:00


In [ ]:
!pip install SpeechRecognition

In [9]:
import subprocess
from IPython.display import Audio # mostrara un reproductor dentro de este notebook

# ⬇️abre el archivo audio.wav como entrada
print(subprocess.run(
    "ffmpeg -i audio.wav -af volumedetect -f null - 2>&1 | grep -E 'mean_volume|max_volume'",
    shell=True, capture_output=True, text=True).stdout)
#
# los parametros de (shell = true) guarda lo que imprime en lugar de mostrarlo

Audio("audio.wav")
# ⬆️ llama al archivo y lo representa en la terminal

[Parsed_volumedetect_0 @ 0x56e5f0f6ab00] mean_volume: -30.8 dB
[Parsed_volumedetect_0 @ 0x56e5f0f6ab00] max_volume: -7.9 dB



In [13]:
#Aqui aplicamos solo dos modulos de una solo linea
import subprocess, requests# para hacer peticiones HTTP (descargar archivos, llamar APIs)
        #⬆️permite ejecutarlo en la terminal de colab
        #se usa casi siempre para llamar a ffmpeg para convertirlo a voz
import speech_recognition as sr
            #⬆️ convirte voz a texto
from base64 import b64decode
# ⬆️aqui importamos solo la funcion de b64decode del modulo base64
from IPython.display import Javascript, display, Image
                                                  # ⬆️el de imagenes no hace falta se me paso por alto
                            #⬆️convierte una cadena de texto en un objeto
from google.colab import output
                          # esta linea de coddigo es un puente
        # ⬆️ ejecuta una expresión JavaScript en el navegador y devuelve el resultado a Python
JS = """
var record = t => new Promise(async res => {
  const s = await navigator.mediaDevices.getUserMedia({audio: true});
  const r = new MediaRecorder(s), p = [];
  r.ondataavailable = e => p.push(e.data);
  r.onstop = () => { const f = new FileReader(); f.onloadend = () => res(f.result); f.readAsDataURL(new Blob(p)); };
  r.start(); setTimeout(() => r.stop(), t);
});
"""

def grabar(seg=5):
  # definimos la funcion de con un parametro de (seg) osea segundos, lo deje por defecto
    display(Javascript(JS)) # inyectamos un simple java scritp en la pagina de colab
                            # record no existe en el navegador, queda definida y lista para usarse
    print(f"🎙️ Habla ahora ({seg} s)...")# la F y las comillas meten las variables dentro de las llaves
    datos = output.eval_js(f"record({seg * 1000})")
                        # ⬆️ aqui construye el texto que es trabajo en 1000 MILISEGUNDOS
    open("audio.webm", "wb").write(b64decode(datos.split(",")[1]))# toma la segunda parte, solo el base64 sin el encabezado.
    subprocess.run("ffmpeg -y -i audio.webm -ar 16000 -ac 1 audio.wav -loglevel quiet", shell=True)
      # (-y)	sobrescribe audio.wav si ya existe, sin preguntar
def reconocer(codigo):
    r = sr.Recognizer() # crea el objeto que hace el reconocimiento.
    with sr.AudioFile("audio.wav") as source: # abre el archivo que creó grabar
        audio = r.record(source) # carga todo el audio en memoria
    try:
        return r.recognize_google(audio, language=codigo)# envía el audio a Google y devuelve el texto
    except sr.UnknownValueError: # Google recibió el audio pero no entendió nada
      # try / except significa "intenta esto, y si falla de cierta forma, haz esto otro"Así el programa no se rompe
        print("No se pudo entender el audio / Could not understand audio")
    except sr.RequestError as e: # no hubo conexión con el servicio de Google.
        print("No se pudo conectar con Google Speech Recognition; {0}".format(e))
    return None

def buscar(texto, lang):# ⬇️hace la petición web.
    datos = requests.get(
        f"https://{lang}.wikipedia.org/w/api.php",
        params={"action": "query", "generator": "search", "gsrsearch": texto,
                "gsrlimit": 1, "prop": "pageimages|info|extracts", "inprop": "url",
                "piprop": "thumbnail", "pithumbsize": 500, "exintro": 1,
                "explaintext": 1, "exsentences": 3, "format": "json"},
        headers={"User-Agent": "practica-voz/1.0"}
    ).json()# .json() convierte la respuesta en un diccionario de Python.
    paginas = datos.get("query", {}).get("pages")
    # .get("pages") toma las páginas encontradas, que vienen como un diccionario con el ID de página como clave.
    return list(paginas.values())[0] if paginas else None
    # list(paginas.values())[0] toma el primer valor, que es la página.


    #          -----------Explicacion de Sintaxis de la funcion [Buscar]------------
    #          action: query	hacer una consulta
    #             generator: search	buscar páginas por texto
    #                 gsrsearch: texto	lo que dijiste
    #                     gsrlimit: 1	solo el primer resultado
    #                       prop: pageimages|info|extracts	pide la imagen, la información (URL) y el resumen
    #                     inprop: url	incluye el enlace de la página (fullurl)
    #                 piprop: thumbnail y pithumbsize: 500	miniatura de 500 px de ancho
    #             exintro: 1	solo la introducción del artículo
    #          explaintext: 1	texto plano, sin HTML
    #           exsentences: 3	solo 3 oraciones
    #             format: json	formato de respuesta


# ---------- MENÚ PRINCIPAL ----------
while True:
    print("\n====== ⚒️MENÚ⚒️ ======")
    print("*escribe un numero*")
    print("1. español")
    print("2. english")
    print("3. salir")
    op = input("Elige el idioma: ") # guarda lo que escribes.

    if op == "3":
        print("¡Hasta luego!") # if op == "3": ... break sale del bucle. if op not in
        break
    if op not in ("1", "2"):# muestra el error y vuelve al inicio del bucle.
        print("Opción no válida")
        continue

    codigo, lang = ("es-ES", "es") if op == "1" else ("en-US", "en")
    #⬆️asigna dos variables a la vez. Es un if en una línea: si elegiste 1 usa español, y si no, inglés.

    grabar() # ⬅️graba 5 segundos y crea audio.wav.
    texto = reconocer(codigo) # lo transcribe. Si devuelve None, if
    if not texto: # vuelve al menú.
        continue
    print("Dijiste:", texto)

    print("\n¿Qué deseas ver?")
    print("1. Imagen en internet")
    print("2. Artículo")
    eleccion = input("Elige: ")# eliges imagen (1) o artículo (2).

    pagina = buscar(texto, lang) # busca en Wikipedia lo que dijiste.
    if not pagina:
        print("No se encontró nada sobre ese tema.")
    elif eleccion == "1":
        if "thumbnail" in pagina:
            display(Image(url=pagina["thumbnail"]["source"]))# saca la imagen des una API
        else:
            print("Ese tema no tiene imagen.")
    elif eleccion == "2":
        print("\n" + pagina["title"])# \n salto de pagina
        print(pagina.get("extract", "Sin resumen."))
        print("🔗", pagina["fullurl"])
    else:
        print("Opción no válida")

#         ------------------Cadena de if y elif-------------------
#         La cadena if / elif / else:
#         Si no hay página: avisa que no encontró nada.
#         Si elegiste 1: muestra la imagen con display(Image(url=...)), o avisa si ese tema no tiene.
#         Si elegiste 2: imprime título, resumen y enlace.



====== ⚒️MENÚ⚒️ ======
*escribe un numero*
1. español
2. english
3. salir
Elige el idioma: 3
¡Hasta luego!
